# Tutorial 10: Geometric Independence and Tomography

One of the core design philosophies of `pygeoinf` is that the mathematics of an inversion should be independent of the underlying geometry. Whether you are solving a problem on a 1D line, a 2D periodic torus, or a 3D volume, the interfaces for priors, forward operators, and solvers remain identical.

In this tutorial, we will build a **Geographic Tomography** engine that can be swapped between five different manifolds:
* **Sphere**: Global-scale modeling.
* **Torus**: Periodic rectangular domains.
* **Plane**: Compact Cartesian domains with tapered boundaries.
* **Circle/Line**: 1D periodic and non-periodic intervals.

We will demonstrate how to set up a path-average (ray-tracing) operator and solve for a hidden model using a Woodbury-preconditioned Conjugate Gradient solver.

In [ ]:
import urllib.request
url = "https://raw.githubusercontent.com/da380/pygeoinf/main/tutorials/colab_bootstrap.py"
exec(urllib.request.urlopen(url).read())
setup_env(extra="sphere")

import numpy as np
import matplotlib.pyplot as plt
import pygeoinf as inf

# For reproducibility
np.random.seed(42)

# Global Tomography Parameters
order = 2           # Sobolev smoothness order
scale = 0.2         # Sobolev length scale
prior_scale = 0.1   # Standard deviation of the prior
n_sources = 12
n_receivers = 12

print("Global parameters initialized.")

## 1. Selecting the Geometry

The power of the abstract `SymmetricSpace` interface is that we can initialize any manifold and the rest of the code won't need to know which one we picked. 

**Task:** Change the `GEOMETRY` variable below to explore how the same inversion logic adapts to different topologies.

In [ ]:
# CHOOSE: "Sphere", "Torus", "Plane", "Circle", or "Line"
GEOMETRY = "Plane" 

if GEOMETRY == "Torus":
    from pygeoinf.symmetric_space.torus import Sobolev, plot, plot_geodesic_network
    model_space = Sobolev.from_heat_kernel_prior(
        prior_scale, order, scale, radius_x=1.0, radius_y=1.0, power_of_two=True, min_degree=32
    )

elif GEOMETRY == "Plane":
    from pygeoinf.symmetric_space.plane import Sobolev, plot, plot_geodesic_network
    model_space = Sobolev.from_heat_kernel_prior(
        prior_scale, order, scale, ax=0.0, bx=6.0, cx=0.5, ay=0.0, by=6.0, cy=0.5, power_of_two=True, min_degree=32
    )

elif GEOMETRY == "Sphere":
    from pygeoinf.symmetric_space.sphere import Sobolev, plot, plot_geodesic_network, create_map_figure
    model_space = Sobolev.from_heat_kernel_prior(
        prior_scale, order, scale, radius=1.0, power_of_two=True, min_degree=32
    )

elif GEOMETRY == "Circle":
    from pygeoinf.symmetric_space.circle import Sobolev, plot
    model_space = Sobolev.from_heat_kernel_prior(prior_scale, order, scale, radius=1.0, power_of_two=True, min_degree=32)
    plot_geodesic_network = None 

elif GEOMETRY == "Line":
    from pygeoinf.symmetric_space.line import Sobolev, plot
    model_space = Sobolev.from_heat_kernel_prior(prior_scale, order, scale, a=0.0, b=10.0, power_of_two=True, min_degree=32)
    plot_geodesic_network = None

print(f"Initialized {GEOMETRY} with dimension {model_space.dim}")

## 2. The Forward Problem: Ray Tomography

In a tomographic experiment, we don't observe the function $u$ directly. Instead, we measure its average value along a series of paths $\gamma$ (rays):
$$\text{data}_i =  \int_{\gamma_i} u(s) ds + \epsilon_i$$
where $\epsilon_i$ is measurement noise.

Because `pygeoinf` is geometry-aware, the `path_average_operator` automatically calculates these integrals correctly, whether the paths are straight lines on a plane or great-circle geodesics on a sphere.

In [ ]:
# 1. Generate random paths (sources and receivers)
# model_space.random_point() handles 1D floats or 2D tuples automatically
receivers = model_space.random_points(n_receivers)
sources = model_space.random_points(n_sources)
paths = [(src, rec) for src in sources for rec in receivers]

# 2. Define the Forward Operator and Data Noise
forward_operator = model_space.path_average_operator(paths) #
data_error_measure = inf.GaussianMeasure.from_standard_deviation(
    forward_operator.codomain, 0.05
) #

# 3. Combine into a Linear Forward Problem
forward_problem = inf.LinearForwardProblem(
    forward_operator, data_error_measure=data_error_measure
) #

# 4. Define the Prior
model_prior = model_space.point_value_scaled_heat_kernel_gaussian_measure(prior_scale) #

print(f"Forward problem set up with {len(paths)} ray paths.")

## 3. Synthetic Data Generation

To test our inversion, we need synthetic "ground truth" and corresponding noisy data. We can generate these by sampling from the **joint measure** of the prior and the likelihood. This ensures our synthetic model is statistically consistent with our prior assumptions.

In [ ]:
# Generate a consistent (model, data) pair
model_true, data_obs = forward_problem.joint_measure(model_prior).sample() #

print("Synthetic model and data generated.")

## 4. Solving the Bayesian Inverse Problem

Now we solve for the **posterior distribution** of the model given the data. 

To handle the high dimensionality of 2D manifolds, we use a **Surrogate Preconditioner**. This builds a low-resolution version of the problem to guide the Conjugate Gradient (CG) solver, drastically reducing the number of iterations required to find the solution.

In [ ]:
# 1. Initialize the Bayesian Inversion
inverse_problem = inf.LinearBayesianInversion(forward_problem, model_prior) #

# 2. Build the Woodbury Surrogate Preconditioner
# We use a space with 1/4 the resolution as a surrogate
surrogate_space = model_space.with_degree(model_space.degree // 4)
raw_surrogate_prior = surrogate_space.point_value_scaled_heat_kernel_gaussian_measure(
    prior_scale
)
woodbury_solver = inf.LUSolver()
damped_surrogate_prior = raw_surrogate_prior.with_regularized_inverse(
    woodbury_solver, damping=1e-6
)
precon = inverse_problem.surrogate_inversion(
    alternate_forward_operator=surrogate_space.path_average_operator(paths),
    alternate_prior_measure=damped_surrogate_prior,
).woodbury_data_preconditioner(woodbury_solver)


# 3. Solve for the Posterior Expectation
solver = inf.CGMatrixSolver() #
model_posterior = inverse_problem.model_posterior_measure(
    data_obs, solver, preconditioner=precon
) #

print(f"Solution found in {solver.iterations} iterations.")

# Estimate uncertainty (pointwise standard deviation)
posterior_std = model_posterior.sample_pointwise_std(100) #

## 5. Visualizing Across Geometries

Finally, we visualize the results. We use a helper function to bridge the small API differences between 1D line plots and 2D heatmaps, ensuring our tutorial remains geometry-agnostic.

In [ ]:
def plot_unified(field, title, label, cmap="RdBu_r", symmetric=False):
    """Wraps API differences with slimmed-down colorbars."""
    if GEOMETRY == "Sphere":
        fig, ax = create_map_figure(figsize=(8, 5))
    else:
        # constrained_layout handles the spacing between title, plot, and bar
        fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")

    # Common colorbar settings to keep things consistent
    cbar_opts = {
        "label": label, 
        "orientation": "horizontal", 
        "pad": 0.08,   # Space between plot and bar
        "shrink": 0.7  # Shrinks the width of the bar
    }

    if GEOMETRY == "Sphere":
        plot(field, ax=ax, colorbar=True, symmetric=symmetric, cmap=cmap, 
             colorbar_kwargs=cbar_opts)
        if plot_geodesic_network:
            plot_geodesic_network(paths, ax=ax, alpha=0.1, color="black")

    elif GEOMETRY in ["Circle", "Line"]:
        plot(model_space, field, ax=ax, color="blue", label=label)
        ax.set_ylabel("Amplitude")

    else:
        # Torus / Plane
        plot(model_space, field, ax=ax, colorbar=True, symmetric=symmetric, cmap=cmap,
             colorbar_kwargs=cbar_opts)
        if plot_geodesic_network:
            plot_geodesic_network(paths, ax=ax, alpha=0.1, color="black")

    ax.set_title(title)
    return ax

# Visualize results
plot_unified(model_true, f"True Model ({GEOMETRY})", "Truth", symmetric=True)
plot_unified(model_posterior.expectation, "Posterior Expectation", "Mean", symmetric=True)

# Visualize Uncertainty
if GEOMETRY in ["Circle", "Line"]:
    from pygeoinf.symmetric_space.line import plot_error_bounds #
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_error_bounds(model_space, model_posterior.expectation, posterior_std, ax=ax, alpha=0.3, color="blue")
    plot(model_space, model_posterior.expectation, ax=ax, color="black", label="Mean")
    ax.set_title(f"Uncertainty on {GEOMETRY}")
    plt.show()
else:
    plot_unified(posterior_std, "Posterior Standard Deviation", "Uncertainty", cmap="viridis")